# Revolut FAQ RAG Chatbot

A minimal single-turn RAG chatbot over Revolut help articles.

**Stack:** `openai` for embeddings + chat, `numpy` for similarity search, `json` for loading. No frameworks, no vector DB — everything in memory.

In [1]:
%pip install -q openai numpy

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import asyncio
import os
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI, AsyncOpenAI
from tqdm.asyncio import tqdm
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import config - single source of truth
import sys
sys.path.insert(0, str(Path(os.getcwd()).parent))
from src.config import *

print(f"Using EMBED_MODEL: {EMBED_MODEL}")
print(f"Using CHAT_MODEL: {CHAT_MODEL}")
print(f"Using TOP_K: {TOP_K}")

# Validate articles path exists
assert ARTICLES_PATH.exists(), f"Articles file not found: {ARTICLES_PATH}"
print(f"Articles path: {ARTICLES_PATH}")

# Initialize clients
client = OpenAI(api_key=OPENAI_API_KEY)
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# Load system prompt from file
with open(RAG_SYSTEM_PROMPT_PATH, 'r') as f:
    SYSTEM_PROMPT = f.read().strip()
print(f"Loaded system prompt from {RAG_SYSTEM_PROMPT_PATH}")

Using EMBED_MODEL: text-embedding-3-small
Using CHAT_MODEL: gpt-3.5-turbo
Using TOP_K: 4
Articles path: /Users/veniamin/Projects/chatbot-evals-ai/data/revolut_help_articles.jsonl
Loaded system prompt from /Users/veniamin/Projects/chatbot-evals-ai/prompts/rag_system.txt


## 1. Load articles

In [3]:
articles = []
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        articles.append(json.loads(line))

print(f"Loaded {len(articles)} articles from {ARTICLES_PATH}")
print("Example:", articles[0]["title"])

Loaded 786 articles from /Users/veniamin/Projects/chatbot-evals-ai/data/revolut_help_articles.jsonl
Example: How can I see my cashflow analytics?


## 2. Embed all articles

We embed `title + content_text` so the title contributes to retrieval. Batched to keep things fast.

In [4]:
def article_to_text(a):
    return f"{a['title']}\n\n{a['content_text']}"

def embed_texts(texts, batch_size=100):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return np.array(out, dtype=np.float32)

texts = [article_to_text(a) for a in articles]
embeddings = embed_texts(texts)

# L2-normalize once so cosine similarity is just a dot product
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (786, 1536)


## 3. Retrieval

In [5]:
def retrieve(query, k=TOP_K):
    q_emb = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    q_vec = np.array(q_emb, dtype=np.float32)
    q_vec = q_vec / np.linalg.norm(q_vec)

    scores = embeddings @ q_vec
    top_idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), articles[i]) for i in top_idx]

## 4. Single-turn chat

In [6]:
# SYSTEM_PROMPT is now loaded from prompts/rag_system.txt via config

def format_context(hits):
    parts = []
    for rank, (idx, score, art) in enumerate(hits, start=1):
        parts.append(
            f"[Article {rank}] {art['title']}\n{art['content_text']}"
        )
    return "\n\n---\n\n".join(parts)

def ask(question, k=TOP_K):
    """Sync ask function - thin wrapper over async_answer_with_context."""
    answer, context, hits = asyncio.run(async_answer_with_context(question, k=k))
    return answer, hits

async def async_answer_with_context(query, k=TOP_K):
    """
    Async RAG query that returns answer, extracted context, and hits.
    Uses AsyncOpenAI for both embedding and chat calls.
    """
    # Async embedding
    q_emb_resp = await async_client.embeddings.create(
        model=EMBED_MODEL,
        input=[query]
    )
    q_emb = q_emb_resp.data[0].embedding
    q_vec = np.array(q_emb, dtype=np.float32)
    q_vec = q_vec / np.linalg.norm(q_vec)
    
    # Retrieval
    scores = embeddings @ q_vec
    top_idx = np.argsort(-scores)[:k]
    hits = [(int(i), float(scores[i]), articles[i]) for i in top_idx]
    
    # Format context
    context = format_context(hits)
    
    # Async chat completion
    user_msg = (
        f"Help articles:\n\n{context}\n\n"
        f"Question: {query}"
    )
    
    resp = await async_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
    )
    answer = resp.choices[0].message.content
    
    return answer, context, hits

## 5. Try it

In [7]:
question = "как открыть аккаунт в монзо"

# Test async_answer_with_context
answer, context, hits = await async_answer_with_context(question)

print("Q:", question)
print("\nA:", answer)
print("\nContext (first 200 chars):", context[:200] + "...")
print("\nRetrieved articles:")
for rank, (idx, score, art) in enumerate(hits, start=1):
    print(f"  {rank}. [{score:.3f}] {art['title']}")

Q: как открыть аккаунт в монзо

A: I don't know.

Context (first 200 chars): [Article 1] Open a Revolut – Kids & Teens account
## Create an account for your kids or teens
In the main Revolut app: 
- Go to 'Home' on the bottom menu
- Below your balance, tap Accounts
- Tap 'Add ...

Retrieved articles:
  1. [0.369] Open a Revolut – Kids & Teens account
  2. [0.357] Duplicate account
  3. [0.337] Open an investment account
  4. [0.330] Change or verify email address


## 6. Evaluate Synthetic Dataset

Evaluate the RAG assistant on the synthetic dataset of 1500 queries.

In [8]:
# RAG evaluation settings
DATASET_PATH = DATA_DIR / "synthetic_revolut_queries.csv"
RAG_OUTPUT_PATH = DATA_DIR / "synthetic_revolut_rag_outputs.csv"
SAVE_EVERY = 25
RAG_CONCURRENCY = 8
MAX_EVAL_ROWS = None  # FULL DATASET - 1500 queries
ROW_KEY = ["persona", "scenario", "modifier", "query"]

# Load queries
df_queries = pd.read_csv(DATASET_PATH)
print(f"Loaded {len(df_queries)} synthetic queries")
df_queries.head()

Loaded 1500 synthetic queries


,persona,scenario,modifier,query
0,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,empty,I see an ATM withdrawal on my account that I d...
1,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,calm_at_home,A suspicious ATM withdrawal appeared today tha...
2,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,panic_security_fear,urgent! atm cash withdrawal just appeared on m...
3,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,angry_after_waiting,JUST SAW ATM withdrawal I DIDN'T MAKE! Need th...
4,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,confused_by_app_updates,There's an ATM withdrawal on my account that I...


In [9]:
import fcntl
import tempfile

def save_rows(df, path):
    """Atomic write: tmp + os.replace."""
    tmp_path = path.with_suffix('.tmp')
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)

async def answer_dataset_row(row):
    """RAG answer for a single dataset row. Returns dict with exact 7 spec columns."""
    answer, context, hits = await async_answer_with_context(row["query"])
    return {
        "persona": row["persona"],
        "scenario": row["scenario"],
        "modifier": row["modifier"],
        "query": row["query"],
        "answer": answer,
        "extracted_context": context,  # Same format_context(hits) string sent to model
        "retrieved_articles": " | ".join([art["title"] for _, _, art in hits])
    }

async def run_rag_dataset(df_queries, output_path, row_key, max_eval_rows=None):
    """Run RAG evaluation with resume and checkpointing."""
    lock_path = DATA_DIR / ".rag_eval.lock"
    lock_fd = None
    
    # Single-writer lock
    try:
        lock_fd = open(lock_path, 'w')
        fcntl.flock(lock_fd.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        lock_fd.write(str(os.getpid()))
        lock_fd.flush()
    except (IOError, BlockingIOError):
        raise RuntimeError("RAG evaluation already running (lock held)")
    
    try:
        # Load existing output
        if output_path.exists():
            df_existing = pd.read_csv(output_path)
            # Dedupe by key, keep first complete
            df_existing = df_existing.drop_duplicates(subset=row_key, keep='first')
            done_keys = set(zip(*[df_existing[k] for k in row_key]))
            print(f"RAG resume: {len(done_keys)} done, {len(df_queries) - len(done_keys)} missing")
        else:
            df_existing = pd.DataFrame(columns=row_key + ["answer", "extracted_context", "retrieved_articles"])
            done_keys = set()
            print(f"RAG resume: 0 done, {len(df_queries)} missing")
        
        # Filter to missing rows
        df_todo = df_queries[
            ~df_queries.apply(lambda r: tuple(r[k] for k in row_key) in done_keys, axis=1)
        ]
        
        if max_eval_rows:
            df_todo = df_todo.head(max_eval_rows)
        
        if len(df_todo) == 0:
            print("No rows to process - all done")
            return df_existing
        
        print(f"Processing {len(df_todo)} queries...")
        
        # Process with concurrency control
        semaphore = asyncio.Semaphore(RAG_CONCURRENCY)
        results = []
        
        async def process_one(row):
            async with semaphore:
                return await answer_dataset_row(row)
        
        tasks = [process_one(row) for _, row in df_todo.iterrows()]
        
        for i, future in enumerate(tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="RAG")):
            result = await future
            results.append(result)
            
            # Checkpoint
            if (i + 1) % SAVE_EVERY == 0:
                df_checkpoint = pd.concat([df_existing, pd.DataFrame(results)], ignore_index=True)
                save_rows(df_checkpoint, output_path)
                print(f"Checkpoint: {i + 1}/{len(tasks)}")
        
        # Final save
        df_final = pd.concat([df_existing, pd.DataFrame(results)], ignore_index=True)
        save_rows(df_final, output_path)
        print(f"Saved {len(df_final)} results to {output_path}")
        
        return df_final
        
    finally:
        if lock_fd:
            fcntl.flock(lock_fd.fileno(), fcntl.LOCK_UN)
            lock_fd.close()
            if lock_path.exists():
                lock_path.unlink()

# Run evaluation
df_results = await run_rag_dataset(df_queries, RAG_OUTPUT_PATH, ROW_KEY, max_eval_rows=MAX_EVAL_ROWS)
print(f"Total results: {len(df_results)}")
df_results.head()

RAG resume: 25 done, 1475 missing
Processing 1475 queries...


RAG:   0%|          | 0/1475 [00:00<?, ?it/s]

RAG:   0%|          | 1/1475 [00:00<21:03,  1.17it/s]

RAG:   0%|          | 2/1475 [00:01<11:15,  2.18it/s]

RAG:   0%|          | 3/1475 [00:01<08:14,  2.98it/s]

...

RAG: 100%|█████████▉| 1473/1475 [03:52<00:00,  7.53it/s]

RAG: 100%|██████████| 1475/1475 [03:53<00:00,  3.66it/s]

RAG: 100%|██████████| 1475/1475 [03:53<00:00,  6.32it/s]

Checkpoint: 1475/1475
Saved 1500 results to /Users/veniamin/Projects/chatbot-evals-ai/data/synthetic_revolut_rag_outputs.csv
Total results: 1500


,persona,scenario,modifier,query,answer,extracted_context,retrieved_articles
0,eu_freelancer_traveling_uae_male_29,fraud_unrecognised_card_payment,poor_internet_connection,noticed a suspicious card payment from yesterd...,If you noticed a suspicious card payment that ...,[Article 1] My card payment was declined by th...,My card payment was declined by the security s...
1,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,angry_after_waiting,JUST SAW ATM withdrawal I DIDN'T MAKE! Need th...,To report a fraudulent ATM withdrawal you didn...,[Article 1] Why is my ATM withdrawal pending?\...,Why is my ATM withdrawal pending? | My card ha...
2,eu_freelancer_traveling_uae_male_29,fraud_unrecognised_card_payment,empty,I saw a card payment I don't recognize. How ca...,To report a card payment you don't recognize a...,[Article 1] Report a card payment as a scam or...,Report a card payment as a scam or fraud | My ...
3,eu_freelancer_traveling_uae_male_29,fraud_unrecognised_card_payment,calm_at_home,I noticed an unexpected charge on my Revolut a...,To report an unexpected charge on your Revolut...,[Article 1] Report a fraudulent ATM withdrawal...,Report a fraudulent ATM withdrawal | Report a ...
4,eu_freelancer_traveling_uae_male_29,fraud_unauthorised_atm_withdrawal,vague_first_message,unexpected atm withdrawal in dubai just now bu...,To report an unexpected ATM withdrawal that wa...,[Article 1] Report a fraudulent ATM withdrawal...,Report a fraudulent ATM withdrawal | My card h...


In [10]:
# Gate G4 - Verify RAG outputs
print("="*60)
print("Gate G4 - RAG Output Validation")
print("="*60)

df_final = pd.read_csv(RAG_OUTPUT_PATH)

# Row count
print(f"Row count: {len(df_final)} (expected: 1500)")

# Columns
expected_cols = ["persona", "scenario", "modifier", "query", "answer", "extracted_context", "retrieved_articles"]
print(f"Columns correct: {list(df_final.columns) == expected_cols}")

# Null checks
null_answers = df_final["answer"].isnull().sum()
null_context = df_final["extracted_context"].isnull().sum()
print(f"Null answers: {null_answers}")
print(f"Null context: {null_context}")

# Retrieved articles
empty_retrieved = (df_final["retrieved_articles"].str.len() == 0).sum()
print(f"Empty retrieved_articles: {empty_retrieved} ({empty_retrieved/len(df_final)*100:.1f}%)")

# Sample outputs
print(f"\nSample rows:")
for i, row in df_final.sample(3).iterrows():
    print(f"\n[{row['modifier'][:15]:15}] Query: {row['query'][:60]}...")
    print(f"Answer: {row['answer'][:80]}...")
    print(f"Retrieved: {row['retrieved_articles'][:100]}...")

# Final verdict
g4_pass = (len(df_final) == 1500 and 
           list(df_final.columns) == expected_cols and
           null_answers == 0 and 
           empty_retrieved / len(df_final) <= 0.01)

print(f"\n{'✅ Gate G4 PASSED' if g4_pass else '❌ Gate G4 FAILED'}")

Gate G4 - RAG Output Validation


Row count: 1500 (expected: 1500)
Columns correct: True
Null answers: 0
Null context: 0
Empty retrieved_articles: 0 (0.0%)

Sample rows:

[panic_security_] Query: Revolut Terminal not processing payments! Customers waiting ...
Answer: Based on the provided help articles, the issue with the Revolut Terminal not pro...
Retrieved: Failed payments on the Revolut Terminal | I'm having trouble using my Revolut Terminal | Cards and c...

[confused_by_app] Query: I want to move CHF to EUR with schedule, but don't see fee i...
Answer: You can find information about fees for international transfers, including CHF t...
Retrieved: Schedule a transfer | Why was my transfer sent via an intermediary bank? | Bank transfer fees | Mana...

[rushing_with_ty] Query: how is interest added on savings is it taxed like what rates...
Answer: To understand how interest is added on savings, whether it is taxed, and what ra...
Retrieved: Earning interest with Savings | Tax on Savings | How to calculate interest on

## Milestone 0 Verification

Verify .gitignore excludes .env but not data CSVs:

In [11]:
# Check .gitignore status
gitignore_path = Path(os.getcwd()).parent.parent / ".gitignore"
with open(gitignore_path) as f:
    gitignore = f.read()
print(".env in .gitignore:", ".env" in gitignore)
print("data/*.csv in .gitignore:", "data/*.csv" in gitignore)
print("First 20 lines of .gitignore:")
print('\n'.join(gitignore.split('\n')[:20]))

.env in .gitignore: True
data/*.csv in .gitignore: False
First 20 lines of .gitignore:
node_modules
.next
out
dist
.env*
.DS_Store
*.log
*.sqlite
coverage

# local archives
*.zip
*.tar.gz

